# Analyze year-to-year population change by sex in Aruba

This notebook visualizes year-to-year population change between sex in Aruba between 2016 and 2023, establishing a baseline before examining migration drivers and age structure in subsequent notebooks.
This analysis is part of a multi-notebook project that aims to investigate Aruba's dependencies and vulnerabilities, examining health data, economic indicators, and social structures alongside demographic trends.

## Data Source
- **Source: Central Bureau of Statistics Aruba**
- **Dataset:** Key demographic aspects.xlsx
- **Period:** 2015–2023 (Note: 2015 has no baseline and is excluded from analysis)

## Method
Establish baseline population trends before examining population change
and migration drivers in subsequent notebooks.

## Overview

This notebook is structured as follows:

1. **Setup & Configuration** — Import libraries and establish file paths
2. **Data Loading & Exploration** — Load CBS datasets and examine their structure
3. **Analysis & Visualization** — Transform data, calculate trends, and visualize findings

---
**Import libraries and Path**
---

In [1]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# Import Class
from pathlib import Path

# Set Path to current working directory
cwd = Path.cwd()

# Set filesystem paths
ROOT = cwd.parent if cwd.name == "working" else cwd

# Establish paths
ROOT

DATA_RAW = ROOT / "data" / "raw"

DATA_PROCESSED = ROOT / "data" / "processed"

FIGURES = ROOT / "figures"

# Set scope in years of available data
YEAR_MIN = 2015
YEAR_MAX = 2023

In [2]:
# Verify all paths
print("ROOT:", ROOT)
print("RAW DATA:", DATA_RAW)
print("PROCESSED DATA:", DATA_PROCESSED)
print("FIGURES:", FIGURES)

ROOT: /home/ggirelli/Documents/DataAnalysis/Projects/Capstone_1/notebooks
RAW DATA: /home/ggirelli/Documents/DataAnalysis/Projects/Capstone_1/notebooks/data/raw
PROCESSED DATA: /home/ggirelli/Documents/DataAnalysis/Projects/Capstone_1/notebooks/data/processed
FIGURES: /home/ggirelli/Documents/DataAnalysis/Projects/Capstone_1/notebooks/figures


# **Load data set and test for errors**

In [3]:
# Load raw data
RAW_DEMOGRAPHIC_ASPECTS_FILE = DATA_RAW / "Demographic-aspects-2023.xlsx"

In [4]:
# Create "if statement" to test for errors
if not RAW_DEMOGRAPHIC_ASPECTS_FILE.exists():
    raise FileNotFoundError

FileNotFoundError: 

In [ ]:
# Set variable for wide format
demographics_raw_wide = pd.read_excel(RAW_DEMOGRAPHIC_ASPECTS_FILE, header=1)

In [ ]:
indicator_columns = demographics_raw_wide["Key Demographic aspects"]

# Inspect columns
indicator_columns

# **Create variables**

In [ ]:
# Inspect and filter the data into our two groups
males = demographics_raw_wide[demographics_raw_wide["Key Demographic aspects"] == "Males"]
females = demographics_raw_wide[demographics_raw_wide["Key Demographic aspects"] == "Females"]

In [ ]:
# DataFrame variables males
yearly_change_males_df = males.loc[:, 2015:2023].diff(axis=1)
yearly_change_males_df = yearly_change_males_df.drop(columns=[2015])

# DataFrame variables females
yearly_change_females_df = females.loc[:, 2015:2023].diff(axis=1)
yearly_change_females_df = yearly_change_females_df.drop(columns=[2015])

# **Create array variables**

In [ ]:
# Create array variables
yearly_change_males_array = yearly_change_males_df.iloc[0].values
yearly_change_females_array = yearly_change_females_df.iloc[0].values

# **Create time value variables**

In [ ]:
# Combine them into one comparison table
sex_comparison = pd.concat([yearly_change_males_df, yearly_change_females_df], keys=["Males", "Females"])

# Clean up: drop the extra index level
sex_comparison = sex_comparison.droplevel(1)

# Convert values and assign new variables
sex_comparison.columns = sex_comparison.columns.astype(int)

# Variables to plot year values on chart
years = sex_comparison.columns

---
**Create visualization**
---

In [ ]:
from matplotlib.patches import Rectangle

##### Create plot dimension 
plt.figure(figsize=(15, 10))

ax = plt.gca()
ax.spines["left"].set_visible(False)
#####

##### Plot
ax.grid(True, alpha=0.25)

plt.figtext(
    0.125,
    0.02,
    "Source: Aruba Central Bureau of Statistics (CBS)",
    fontsize=9,
    color="0.45"
)
#####

##### Title
ax.set_title("Annual population change in Aruba by sex (2016–2023)", fontsize=14)
#####

# X-axis
ax.set_xticks([2016, 2018, 2020, 2022])
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))

ax.spines["bottom"].set_linewidth(1.5)
ax.spines["bottom"].set_color("0.3")
ax.tick_params(axis="x" ,labelsize=14, which="major", length=9)
ax.tick_params(axis="x" ,labelsize=14, which="minor", length=6)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# X-axis labels
ax.set_xlim(2016, 2023.6)
ax.text(2023 + 0.1, 119.0, "Males", fontsize=14)

ax.set_xlim(2016, 2023.6)
ax.text(2023 + 0.1, 295.0, "Females", fontsize=14)

# Grey band behind x-axis
rect = Rectangle(
    (0, -0.12),
    1,             # width (full axis)
    0.12,          # height of the band
    transform=ax.transAxes,
    color="0.94",
    zorder=0,
    clip_on=False
)

ax.add_patch(rect)

# Zero baseline
ax.axhline(0, linestyle="--", linewidth=1.4, color="black", alpha=0.8)

# Line-graph values
ax.plot(years, yearly_change_males_array, label="Males", linewidth=2.5)
ax.plot(years, yearly_change_females_array, label="Females", linewidth=2.5)

# Shaded area around '2020' for extra clarity
ax.axvspan(2019.5, 2021.5, alpha=0.15)

# Added label
ax.text(2020.5, 250, "COVID-19 disruption period",
         ha="center", fontsize=10)

# Save figure
plt.savefig(FIGURES / "cbs_demographic_changes-2015-2023.png", dpi=300)

plt.show()